 - 带工具的Agent 的工作流程


In [1]:
from langchain_core.tools import tool

@tool
def calculator(operation: str, a: float, b: float=0) -> str:
    """
    执行基本的数学计算
    参数:
        operation: 运算类型，支持 "add"(加), "subtract"(减), "multiply"(乘), "divide"(除), "sqrt"(平方根)
        a: 第一个数字
        b: 第二个数字

    返回:
        计算结果字符串
    """
    operations = {
        "add": lambda x, y: x + y,
        "subtract": lambda x, y: x - y,
        "multiply": lambda x, y: x * y,
        "divide": lambda x, y: x / y if y != 0 else "错误：除数不能为零", 
        "sqrt":lambda x, y: x ** 0.5,
    }

    if operation not in operations:
        return f"不支持的运算类型：{operation}。支持的类型：add, subtract, multiply, divide"

    try:
        result = operations[operation](a, b)
        print("\t|-工具实际结果：", result)
        return f"{a} {operation} {b} = {result}"
    except Exception as e:
        return f"计算错误：{e}"
    
@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息

    参数:
        city: 城市名称，如"北京"、"上海"

    返回:
        天气信息字符串
    """
    # 模拟天气数据（实际应用中应调用真实API）
    weather_data = {
        "上海": "晴天，温度 15°C，空气质量良好",
        "杭州": "多云，温度 18°C，有轻微雾霾",
        "成都": "阴天，温度 22°C，可能有小雨",
        "南京": "小雨，温度 12°C，湿度较高"
    }

    return weather_data.get(city, f"抱歉，暂时没有{city}的天气数据")


In [2]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
model = init_chat_model(
    model="qwen3.5:4b",          # 你在 Ollama 中下载的模型名称
    model_provider="ollama",      # 关键参数，指定使用 ollama 提供商
    temperature=0.7,              # 其他参数与使用任何其他模型一样
    base_url="http://localhost:11434",  # 可选，如果你的 Ollama 不是默认地址，则需要设置。
    extra_body={
        "top_k": 20,
        "chat_template_kwargs": {"enable_thinking": True},
    }
)

agent = create_agent(
    model=model,
    tools=[calculator, get_weather],
    system_prompt="你是一个有帮助的助手。"
)
response = agent.invoke({
    "messages": [{"role": "user", "content": "25乘以8，再求平方根，等于多少？最后再帮我查下杭州天气"}]
})

	|-工具实际结果： 200.0
	|-工具实际结果： 14.142135623730951


In [3]:
print("\n完整消息历史：")
for i, msg in enumerate(response['messages'], 1):
    print("=" * 80)
    print(f"消息 - {i}: {type(msg)}")
    if hasattr(msg, 'name'):
        print(f"消息名: {msg.name}")
    if hasattr(msg, 'content') and msg.content:
        print(f"内容: {msg.content}")

    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"工具调用:")
        for tc in msg.tool_calls:
            print(f"  - 工具: {tc['name']}")
            print(f"  - 参数: {tc['args']}")


完整消息历史：
消息 - 1: <class 'langchain_core.messages.human.HumanMessage'>
消息名: None
内容: 25乘以8，再求平方根，等于多少？最后再帮我查下杭州天气
消息 - 2: <class 'langchain_core.messages.ai.AIMessage'>
消息名: None
工具调用:
  - 工具: calculator
  - 参数: {'operation': 'multiply', 'a': 25, 'b': 8}
消息 - 3: <class 'langchain_core.messages.tool.ToolMessage'>
消息名: calculator
内容: 25.0 multiply 8.0 = 200.0
消息 - 4: <class 'langchain_core.messages.ai.AIMessage'>
消息名: None
工具调用:
  - 工具: calculator
  - 参数: {'operation': 'sqrt', 'a': 200}
消息 - 5: <class 'langchain_core.messages.tool.ToolMessage'>
消息名: calculator
内容: 200.0 sqrt 0 = 14.142135623730951
消息 - 6: <class 'langchain_core.messages.ai.AIMessage'>
消息名: None
工具调用:
  - 工具: get_weather
  - 参数: {'city': '杭州'}
消息 - 7: <class 'langchain_core.messages.tool.ToolMessage'>
消息名: get_weather
内容: 多云，温度 18°C，有轻微雾霾
消息 - 8: <class 'langchain_core.messages.ai.AIMessage'>
消息名: None
内容: 25乘以8等于200，200的平方根约为14.142135623730951。杭州天气：多云，温度18°C，有轻微雾霾。


In [5]:
# 使用 stream 方法
response = agent.stream({
    "messages": [{"role": "user", "content": "25乘以8，再求平方根，等于多少？最后再帮我查下杭州天气"}]
})
for chunk in response:
    # print(type(chunk))
    # print(chunk)
    print("*" * 80)
    if 'model' in chunk:
        latest_msg = chunk["model"]['messages'][-1]
        
    if "tools" in chunk:
        latest_msg = chunk["tools"]['messages'][-1]
        
    if hasattr(latest_msg, 'content') and latest_msg.content:
        print(f"\t|- 内容: {msg.content}")

    if hasattr(latest_msg, 'tool_calls') and latest_msg.tool_calls:
        print(f"\t|- 工具调用:")
        for tc in latest_msg.tool_calls:
            print(f"\t\t|-工具: {tc['name']}")
            print(f"\t\t|-参数: {tc['args']}")

********************************************************************************
	|- 工具调用:
		|-工具: calculator
		|-参数: {'operation': 'multiply', 'a': 25, 'b': 8}
		|-工具: calculator
		|-参数: {'operation': 'sqrt', 'a': 200}
		|-工具: get_weather
		|-参数: {'city': '杭州'}
	|-工具实际结果： 14.142135623730951
	|-工具实际结果： 200.0
********************************************************************************
	|- 内容: 25乘以8等于200，200的平方根约为14.142135623730951。杭州天气：多云，温度18°C，有轻微雾霾。
********************************************************************************
	|- 内容: 25乘以8等于200，200的平方根约为14.142135623730951。杭州天气：多云，温度18°C，有轻微雾霾。
********************************************************************************
	|- 内容: 25乘以8等于200，200的平方根约为14.142135623730951。杭州天气：多云，温度18°C，有轻微雾霾。
********************************************************************************
	|- 内容: 25乘以8等于200，200的平方根约为14.142135623730951。杭州天气：多云，温度18°C，有轻微雾霾。


In [ ]:
for chunk in response:
    print(chunt)